# 梯子 (GPU) — 語数を増やして 損失の下がり方を測る

使う前に: 左の鍵アイコン(シークレット)に `HF_TOKEN` を追加し、 ノートブックからのアクセスをオンにする。 トークンは 読み取りだけでよい。 ランタイムは「ランタイムのタイプを変更」でGPUを選ぶ。

順番に実行する。 ④の速度確認の結果を貼ってもらってから、 ⑥以降を回す。

In [ ]:
# ① GPU の確認
!nvidia-smi -L
import torch
print('cuda:', torch.cuda.is_available(), torch.__version__)
!nproc && free -g | head -2

In [ ]:
# ② 道具とコード (毎回 最新を取る)
!pip -q install tokenizers huggingface_hub
!rm -rf /content/ai-lab && git clone -q https://github.com/yoojpn/ai-lab /content/ai-lab
!cd /content/ai-lab && git log --oneline -1

In [ ]:
# ③ データ (HF の tok/ 約 2.7GB)。 すでに /tmp/tok があれば飛ばす
import os
if not os.path.exists('/tmp/tok/train.bin'):
    from google.colab import userdata
    from huggingface_hub import snapshot_download
    snapshot_download('Youichi1/ai-lab-data', repo_type='dataset', allow_patterns=['tok/*'], local_dir='/content/hfd', token=userdata.get('HF_TOKEN'))
    !rm -rf /tmp/tok && mv /content/hfd/tok /tmp/tok
!ls /tmp/tok | head -5 && df -h /tmp | tail -1

In [ ]:
# 道具: 1 本 / 複数本を同時に走らせる
import json, subprocess, time
KEYS = ('歩 ', '結果:', '長い文脈', 'Traceback', 'Error', 'CUDA', 'GPU の', '合法', 'パラメータ')
def start(ntr, log_every, name, nval=100000, ntest=200000, nlong=40000, **extra):
    cfg = dict(L=4, n=256, ntr=ntr, nval=nval, ntest=ntest, nlong=nlong, log_every=log_every, feats=[1, 2, 3]); cfg.update(extra)
    f = open('/content/%s.log' % name, 'w')
    return subprocess.Popen(['python3', 'exp_ctx.py', json.dumps(cfg)], cwd='/content/ai-lab/hdc', stdout=f, stderr=subprocess.STDOUT, text=True)
def show(name, last=None):
    ls = [l.rstrip()[:300] for l in open('/content/%s.log' % name) if any(k in l for k in KEYS)]
    print('=====', name); print('\n'.join(ls if last is None else ls[-last:]))
def run_many(jobs):
    ps = [start(**j) for j in jobs]
    for p in ps: p.wait()
    for j in jobs: show(j['name'], last=8)

In [ ]:
# ④ 速度確認 (300万語)。 『CUDA グラフの自己確認』 の行と 『歩/秒』 と 最後の『結果:』 を貼る
run_many([dict(ntr=3000000, log_every=500, name='speed', nval=20000, ntest=20000, nlong=5000)])

In [ ]:
# ⑤ バッチを大きくしても同じ点数になるか (300万語、 2 本を同時に)
# 注意: 2 本を同時に走らせるので 歩/秒 は 1 本のときより遅く出る。 見るのは最後の『結果:』の単独
# 私が『回していい』 と言ってから (先頭の # を消す)
# run_many([dict(ntr=3000000, log_every=200, name='b512'),
#           dict(ntr=3000000, log_every=50, name='b2048', B=2048, lr=0.004, lrs=0.01, warm=50)])

In [ ]:
# ⑥ 本番: 語数 4800万 (私が『回していい』 と言ってから)
# run_many([dict(ntr=48000000, log_every=1000, name='ld48')])
# 進み具合は別のセルで: !tail -n 3 /content/ld48.log